# Delta Lake Hands-On in Databricks

## Making Data Lakes Reliable

This notebook is designed to run directly in **Databricks**.

It starts with the limitations of a plain Data Lake and then focuses on practical Delta Lake features using a realistic e-commerce orders use case.

## Topics covered

- Problems with plain Data Lakes
- Why Delta Lake exists
- Delta tables and the `_delta_log`
- ACID transactions
- Table history
- Time travel
- Update and delete
- Merge / upsert
- Schema enforcement
- Bronze, Silver, Gold Lakehouse architecture
- Real-life Data Engineering use cases


# 1. Plain Data Lake Recap

A Data Lake stores raw and processed data files at scale.

Common storage systems:

```text
AWS S3
Azure Data Lake Storage
Google Cloud Storage
HDFS
```

Common file formats:

```text
CSV
JSON
Parquet
```

A common Data Lake layout is:

```text
data_lake/
  bronze/
  silver/
  gold/
```

## Bronze Layer

Raw data as received from source systems.

## Silver Layer

Cleaned, standardized, and validated data.

## Gold Layer

Business-ready analytics tables.


# 2. Problems with Plain Data Lakes

Plain Data Lakes are powerful, but storing only files and folders creates reliability problems.

## Common problems

| Problem | Real-life example |
|---|---|
| Partial writes | Pipeline fails after writing only half the files |
| No transaction log | Hard to know which files belong to the current table version |
| Hard updates | Payment status changes from pending to success |
| Hard deletes | Invalid or sensitive record must be removed |
| Schema issues | Source system adds a new column unexpectedly |
| No easy rollback | Today's pipeline corrupts the dashboard data |
| Concurrent writes | Two jobs write to the same table at the same time |

## Main question

```text
If Data Lakes are just files and folders, how do we make them reliable like database tables?
```

The answer is **Delta Lake**.


# 3. What is Delta Lake?

Delta Lake is a storage layer that adds reliability to Data Lake files.

Plain Data Lake:

```text
Parquet files + folders
```

Delta Lake:

```text
Parquet files + _delta_log transaction log
```

The `_delta_log` tracks table versions and file changes.

## Simple definition

```text
Delta Lake makes Parquet files behave like reliable database tables.
```

## Delta Lake adds

```text
ACID transactions
transaction log
table history
time travel
updates
deletes
merge / upsert
schema enforcement
schema evolution
```


# 4. Delta Lake Architecture

<svg width="900" height="290" xmlns="http://www.w3.org/2000/svg">
  <rect x="40" y="70" width="230" height="150" rx="12" fill="#eef4ff" stroke="#3b5bdb"/>
  <text x="155" y="105" text-anchor="middle" font-size="18" font-family="Arial">Plain Data Lake</text>
  <text x="155" y="140" text-anchor="middle" font-size="14" font-family="Arial">Parquet files</text>
  <text x="155" y="165" text-anchor="middle" font-size="14" font-family="Arial">Partition folders</text>
  <text x="155" y="190" text-anchor="middle" font-size="14" font-family="Arial">No table history</text>

  <rect x="350" y="70" width="200" height="150" rx="12" fill="#fff4e6" stroke="#f08c00"/>
  <text x="450" y="105" text-anchor="middle" font-size="18" font-family="Arial">Add Delta Lake</text>
  <text x="450" y="140" text-anchor="middle" font-size="14" font-family="Arial">Transaction log</text>
  <text x="450" y="165" text-anchor="middle" font-size="14" font-family="Arial">ACID commits</text>
  <text x="450" y="190" text-anchor="middle" font-size="14" font-family="Arial">Table versions</text>

  <rect x="630" y="70" width="230" height="150" rx="12" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="745" y="105" text-anchor="middle" font-size="18" font-family="Arial">Reliable Table</text>
  <text x="745" y="140" text-anchor="middle" font-size="14" font-family="Arial">Updates / Deletes</text>
  <text x="745" y="165" text-anchor="middle" font-size="14" font-family="Arial">Merge / Upsert</text>
  <text x="745" y="190" text-anchor="middle" font-size="14" font-family="Arial">Time Travel</text>

  <line x1="270" y1="145" x2="350" y2="145" stroke="#333" stroke-width="2" marker-end="url(#arrow)"/>
  <line x1="550" y1="145" x2="630" y2="145" stroke="#333" stroke-width="2" marker-end="url(#arrow)"/>

  <defs>
    <marker id="arrow" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#333"/>
    </marker>
  </defs>
</svg>


# 5. Create a Working Database

This notebook uses a database so all tables are organized together.

The database name includes a simple suffix to reduce conflicts.


In [ ]:
# Databricks / PySpark setup
from pyspark.sql.functions import col, lower, trim, to_date, current_timestamp, sum as spark_sum, count, desc
from delta.tables import DeltaTable

DATABASE_NAME = "delta_lake_class_demo"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}")
spark.sql(f"USE {DATABASE_NAME}")

print("Using database:", DATABASE_NAME)


# 6. Clean Up Previous Runs

Run this cell if you want a fresh start.


In [ ]:
# Clean up previous demo tables
for table_name in [
    "bronze_orders",
    "silver_orders",
    "gold_revenue_by_city",
    "gold_revenue_by_status",
    "plain_parquet_orders_temp"
]:
    spark.sql(f"DROP TABLE IF EXISTS {table_name}")

print("Previous demo tables removed if they existed.")


# 7. Real-Life Scenario: E-Commerce Orders Pipeline

An e-commerce company receives order data every day.

The business wants trusted reports such as:

```text
Revenue by city
Revenue by order date
Completed orders count
Payment/order status changes
```

But real data changes:

```text
New orders arrive daily
Pending orders become completed
Some orders are refunded
Invalid rows may need deletion
Source schema may change
Bad pipeline loads may need rollback
```

This is why Delta Lake is useful.


# 8. Create Day 1 Raw Orders

This represents raw order data arriving from a source system.

Some records are valid, some records are not yet clean.


In [ ]:
day1_orders = spark.createDataFrame([
    (1001, "C001", "2026-08-10", 2500.0, "Completed", "Delhi", "India"),
    (1002, "C002", "2026-08-10", -500.0, "Completed", "Mumbai", "India"),
    (1003, "C003", "2026-08-10", 1800.0, "Pending", "Delhi", "India"),
    (1004, "C004", "2026-08-11", 3200.0, "Completed", "Pune", "India"),
    (1005, "C005", "2026-08-11", 1200.0, "Cancelled", "Mumbai", "India")
], ["order_id", "customer_id", "order_date", "amount", "status", "city", "country"])

day1_orders.show()


# 9. Bronze Delta Table

Bronze stores raw data as received.

In many real pipelines, Bronze may store raw CSV/JSON/Parquet files.

Here we store Bronze as Delta so the raw landing table also has history and reliability.


In [ ]:
day1_orders.write.format("delta").mode("overwrite").saveAsTable("bronze_orders")

spark.sql("SELECT * FROM bronze_orders ORDER BY order_id").show()


# 10. Create Silver Delta Table

Silver contains cleaned and standardized data.

Cleaning rules:

```text
Keep amount > 0
Standardize status to lowercase
Convert order_date to date
Add processed_at timestamp
```


In [ ]:
bronze_orders_df = spark.table("bronze_orders")

silver_orders = (
    bronze_orders_df
    .filter(col("amount") > 0)
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("city", trim(col("city")))
    .withColumn("country", trim(col("country")))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("processed_at", current_timestamp())
)

silver_orders.write.format("delta").mode("overwrite").saveAsTable("silver_orders")

spark.sql("SELECT * FROM silver_orders ORDER BY order_id").show()


# 11. Inspect Delta Table Details

Delta tables are stored as Parquet files plus a transaction log.

Databricks lets us inspect table metadata using SQL.


In [ ]:
spark.sql("DESCRIBE DETAIL silver_orders").show(truncate=False)


# 12. Delta Table History

Every write operation creates a new Delta table version.

This is useful for:

```text
auditing
debugging
rollback
pipeline monitoring
```


In [ ]:
spark.sql("DESCRIBE HISTORY silver_orders").show(truncate=False)


# 13. Real-Life Use Case 1: Append New Daily Orders

Every day, new order data arrives.

With Delta Lake, appending new records creates a new table version.


In [ ]:
day2_orders = spark.createDataFrame([
    (1006, "C006", "2026-08-12", 4500.0, "Completed", "Bangalore", "India"),
    (1007, "C007", "2026-08-12", 7500.0, "Pending", "Dubai", "UAE"),
    (1008, "C008", "2026-08-12", 6200.0, "Completed", "Singapore", "Singapore")
], ["order_id", "customer_id", "order_date", "amount", "status", "city", "country"])

day2_clean = (
    day2_orders
    .filter(col("amount") > 0)
    .withColumn("status", lower(trim(col("status"))))
    .withColumn("order_date", to_date(col("order_date")))
    .withColumn("processed_at", current_timestamp())
)

day2_clean.write.format("delta").mode("append").saveAsTable("silver_orders")

spark.sql("SELECT * FROM silver_orders ORDER BY order_id").show()


# 14. Check History After Append

The append operation created a new table version.


In [ ]:
spark.sql("DESCRIBE HISTORY silver_orders").show(truncate=False)


# 15. Real-Life Use Case 2: Time Travel

Suppose today's pipeline added data and the business wants to compare the table before and after the load.

Delta Lake allows querying older table versions.

## Current version


In [ ]:
spark.sql("SELECT COUNT(*) AS current_row_count FROM silver_orders").show()


## Version 0

Version 0 was the first version of the Silver table.


In [ ]:
spark.sql("SELECT COUNT(*) AS version_0_row_count FROM silver_orders VERSION AS OF 0").show()

spark.sql("SELECT * FROM silver_orders VERSION AS OF 0 ORDER BY order_id").show()


# 16. Real-Life Use Case 3: Update an Order Status

An order can change after the first load.

Example:

```text
order_id = 1007 was pending
later it became completed
```

Plain Parquet does not naturally support table-level updates.

Delta Lake supports updates.


In [ ]:
spark.sql("""
UPDATE silver_orders
SET status = 'completed'
WHERE order_id = 1007
""")

spark.sql("SELECT * FROM silver_orders WHERE order_id = 1007").show()


# 17. Real-Life Use Case 4: Delete Invalid or Test Records

Sometimes data must be removed from trusted tables.

Examples:

```text
test orders
invalid orders
duplicate records
privacy/compliance deletes
```

Delta Lake supports deletes.


In [ ]:
# Add a test order, then delete it

test_order = spark.createDataFrame([
    (9999, "TEST", "2026-08-12", 1.0, "completed", "Test City", "Test Country")
], ["order_id", "customer_id", "order_date", "amount", "status", "city", "country"]) .withColumn("order_date", to_date(col("order_date"))) .withColumn("processed_at", current_timestamp())

test_order.write.format("delta").mode("append").saveAsTable("silver_orders")

print("Before delete:")
spark.sql("SELECT * FROM silver_orders WHERE order_id = 9999").show()

spark.sql("DELETE FROM silver_orders WHERE order_id = 9999")

print("After delete:")
spark.sql("SELECT * FROM silver_orders WHERE order_id = 9999").show()


# 18. Real-Life Use Case 5: Merge / Upsert

Merge is one of the most important Delta Lake features for Data Engineering.

It handles two situations together:

```text
Existing record arrives again with updated values → update
New record arrives for the first time → insert
```

This is also called an **upsert**.

## Scenario

A source system sends order updates:

```text
order_id 1005 changed from cancelled to completed
order_id 1009 is a new order
```


In [ ]:
order_updates = spark.createDataFrame([
    (1005, "C005", "2026-08-11", 1200.0, "completed", "Mumbai", "India"),
    (1009, "C009", "2026-08-13", 9000.0, "completed", "Delhi", "India")
], ["order_id", "customer_id", "order_date", "amount", "status", "city", "country"]) .withColumn("order_date", to_date(col("order_date"))) .withColumn("status", lower(trim(col("status")))) .withColumn("processed_at", current_timestamp())

order_updates.createOrReplaceTempView("order_updates")

order_updates.show()


In [ ]:
spark.sql("""
MERGE INTO silver_orders AS target
USING order_updates AS source
ON target.order_id = source.order_id
WHEN MATCHED THEN UPDATE SET
  target.customer_id = source.customer_id,
  target.order_date = source.order_date,
  target.amount = source.amount,
  target.status = source.status,
  target.city = source.city,
  target.country = source.country,
  target.processed_at = source.processed_at
WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("SELECT * FROM silver_orders ORDER BY order_id").show()


# 19. Check History After Update, Delete, and Merge

The history shows operations performed on the table.


In [ ]:
spark.sql("DESCRIBE HISTORY silver_orders").show(truncate=False)


# 20. Real-Life Use Case 6: Schema Enforcement

A source system may unexpectedly send a new column or wrong schema.

Example:

```text
old schema:
order_id, customer_id, order_date, amount, status, city, country

new unexpected schema:
order_id, customer_id, amount, status, unexpected_column
```

Delta Lake protects the table from accidental schema corruption.


In [ ]:
bad_schema_orders = spark.createDataFrame([
    (2001, "C200", 1000.0, "completed", "unexpected_value")
], ["order_id", "customer_id", "amount", "status", "unexpected_column"])

try:
    bad_schema_orders.write.format("delta").mode("append").saveAsTable("silver_orders")
except Exception as error:
    print("Write failed because schema does not match the Delta table.")
    print(type(error).__name__)


# 21. Schema Evolution Concept

Sometimes schema changes are valid.

Example:

```text
The business adds a new discount_amount column.
```

Schema evolution means allowing a table schema to change in a controlled way.

In production, schema evolution should be used carefully.

A common practice is:

```text
Reject unexpected schema changes by default.
Allow approved schema changes only after review.
```


# 22. Create Gold Delta Table: Revenue by City

Gold tables are business-ready analytics tables.

Now create a Gold table from the Silver Delta table.


In [ ]:
gold_revenue_by_city = spark.sql("""
SELECT
  order_date,
  city,
  country,
  SUM(amount) AS total_revenue,
  COUNT(*) AS completed_order_count
FROM silver_orders
WHERE status = 'completed'
GROUP BY order_date, city, country
ORDER BY order_date, total_revenue DESC
""")

gold_revenue_by_city.show()


In [ ]:
gold_revenue_by_city.write.format("delta").mode("overwrite").saveAsTable("gold_revenue_by_city")

spark.sql("SELECT * FROM gold_revenue_by_city ORDER BY order_date, total_revenue DESC").show()


# 23. Create Another Gold Table: Revenue by Status

This table helps the business understand order status distribution.


In [ ]:
gold_revenue_by_status = spark.sql("""
SELECT
  status,
  COUNT(*) AS order_count,
  SUM(amount) AS total_amount
FROM silver_orders
GROUP BY status
ORDER BY total_amount DESC
""")

gold_revenue_by_status.write.format("delta").mode("overwrite").saveAsTable("gold_revenue_by_status")

gold_revenue_by_status.show()


# 24. Bronze, Silver, Gold with Delta Lake

A practical lakehouse layout:

<svg width="900" height="270" xmlns="http://www.w3.org/2000/svg">
  <rect x="60" y="80" width="190" height="90" rx="12" fill="#fff4e6" stroke="#f08c00"/>
  <text x="155" y="110" text-anchor="middle" font-size="17" font-family="Arial">Bronze</text>
  <text x="155" y="135" text-anchor="middle" font-size="13" font-family="Arial">Raw source data</text>
  <text x="155" y="155" text-anchor="middle" font-size="13" font-family="Arial">Append-friendly</text>

  <rect x="350" y="80" width="190" height="90" rx="12" fill="#f1f3f5" stroke="#495057"/>
  <text x="445" y="110" text-anchor="middle" font-size="17" font-family="Arial">Silver</text>
  <text x="445" y="135" text-anchor="middle" font-size="13" font-family="Arial">Clean Delta tables</text>
  <text x="445" y="155" text-anchor="middle" font-size="13" font-family="Arial">Validated data</text>

  <rect x="640" y="80" width="190" height="90" rx="12" fill="#ebfbee" stroke="#2b8a3e"/>
  <text x="735" y="110" text-anchor="middle" font-size="17" font-family="Arial">Gold</text>
  <text x="735" y="135" text-anchor="middle" font-size="13" font-family="Arial">Business metrics</text>
  <text x="735" y="155" text-anchor="middle" font-size="13" font-family="Arial">BI-ready tables</text>

  <line x1="250" y1="125" x2="350" y2="125" stroke="#333" stroke-width="2" marker-end="url(#arrow2)"/>
  <line x1="540" y1="125" x2="640" y2="125" stroke="#333" stroke-width="2" marker-end="url(#arrow2)"/>

  <defs>
    <marker id="arrow2" markerWidth="10" markerHeight="10" refX="8" refY="3" orient="auto">
      <path d="M0,0 L0,6 L9,3 z" fill="#333"/>
    </marker>
  </defs>
</svg>

## Recommended use

```text
Bronze: raw records and ingestion traceability
Silver: cleaned trusted Delta tables
Gold: reporting and analytics Delta tables
```


# 25. Query Delta Tables with SQL

Delta tables can be queried like normal SQL tables.


In [ ]:
spark.sql("SHOW TABLES").show(truncate=False)


In [ ]:
spark.sql("""
SELECT
  order_date,
  city,
  total_revenue
FROM gold_revenue_by_city
ORDER BY order_date, total_revenue DESC
""").show()


# 26. Delta Lake vs Plain Parquet

| Capability | Plain Parquet | Delta Lake |
|---|---:|---:|
| Efficient columnar storage | Yes | Yes |
| Partitioning | Yes | Yes |
| ACID transactions | No | Yes |
| Transaction log | No | Yes |
| Table history | No | Yes |
| Time travel | No | Yes |
| Update records | Difficult | Supported |
| Delete records | Difficult | Supported |
| Merge / upsert | Difficult | Supported |
| Schema enforcement | Limited | Supported |

## Strong summary

```text
Parquet is a file format.
Delta Lake is a reliable table layer built on top of Parquet files.
```


# 27. Real-Life Mapping of Delta Features

| Situation | Delta Lake feature |
|---|---|
| Pipeline fails halfway | ACID transaction |
| Need to know what changed | Table history |
| Dashboard broke after new load | Time travel |
| Payment status changed | Update |
| Test record must be removed | Delete |
| Daily file contains old and new orders | Merge / upsert |
| Source sends wrong columns | Schema enforcement |
| Approved new column is added | Schema evolution |


# 28. Practice Exercise

## Task

Create a new Delta table called `silver_payments`.

Use this sample payment data:

```text
payment_id, order_id, payment_method, payment_status, payment_date
PMT001, 1001, UPI, success, 2026-08-10
PMT002, 1003, card, pending, 2026-08-10
PMT003, 1004, netbanking, success, 2026-08-11
PMT004, 1007, card, pending, 2026-08-12
```

Then perform:

```text
1. Write the table as Delta
2. Update payment_status for PMT004 to success
3. Insert one new payment record
4. Run DESCRIBE HISTORY
5. Query an older version
```


In [ ]:
# Practice starter

payments = spark.createDataFrame([
    ("PMT001", 1001, "UPI", "success", "2026-08-10"),
    ("PMT002", 1003, "card", "pending", "2026-08-10"),
    ("PMT003", 1004, "netbanking", "success", "2026-08-11"),
    ("PMT004", 1007, "card", "pending", "2026-08-12")
], ["payment_id", "order_id", "payment_method", "payment_status", "payment_date"])

payments_clean = (
    payments
    .withColumn("payment_method", lower(trim(col("payment_method"))))
    .withColumn("payment_status", lower(trim(col("payment_status"))))
    .withColumn("payment_date", to_date(col("payment_date")))
)

payments_clean.show()


In [ ]:
# Write your solution below

# 1. Write payments_clean as a Delta table called silver_payments

# 2. Update payment_status for PMT004 to success

# 3. Insert one new payment record

# 4. Run DESCRIBE HISTORY silver_payments

# 5. Query VERSION AS OF 0


# 29. Interview Questions

## Q1. What problem does Delta Lake solve?

Delta Lake makes Data Lake tables reliable by adding ACID transactions, table history, time travel, schema enforcement, updates, deletes, and merge support.

## Q2. What is `_delta_log`?

`_delta_log` is the transaction log of a Delta table. It tracks table versions and file changes.

## Q3. Why is Delta Lake better than plain Parquet for production pipelines?

Plain Parquet is only a file format. Delta Lake adds table reliability features such as transactions, rollback, updates, deletes, and schema control.

## Q4. What is time travel?

Time travel allows users to query previous versions of a Delta table.

## Q5. What is merge or upsert?

Merge updates existing records and inserts new records in a single operation.

## Q6. How does Delta Lake fit into Bronze, Silver, Gold architecture?

Bronze stores raw data, Silver stores cleaned trusted Delta tables, and Gold stores business-ready Delta tables.

## Q7. Why is schema enforcement important?

Schema enforcement prevents accidental writes with incorrect columns or incompatible data types.

## Q8. How does Delta Lake connect to Databricks?

Delta Lake provides reliable lakehouse tables, and Databricks provides a managed platform to build, query, schedule, and govern those tables.


# 30. Final Summary

## Plain Data Lake

```text
Files and folders
Fast and scalable
But reliability is limited
```

## Delta Lake

```text
Parquet files + transaction log
Reliable table versions
Supports update, delete, merge, history, and time travel
```

## Main takeaway

```text
A Data Lake stores files.
Delta Lake turns those files into trustworthy tables.
```

## Next step

```text
Databricks platform: notebooks, compute, Delta tables, SQL, workflows, and governance.
```
